# Семинар W2. Pandas-1. Введение и базовый анализ данных

Добро пожаловать на практический семинар по основам работы с библиотекой **Pandas**!

## Контекст бизнес-кейса
Вы работаете аналитиком в быстрорастущей сети онлайн-кинотеатров. В отдел маркетинга регулярно поступают выгрузки с логами пользователей (регистрации, оплаты подписок), но эти «сырые» данные часто приходят грязными: с пропусками, аномальными транзакциями и в неудобных для анализа форматах.

**Цель семинара:** Провести первичную гигиену данных, отфильтровать подозрительные оплаты, подготовить списки клиентов для рекламных рассылок и применить базовые функции pandas.

Мы будем работать с имитацией выгрузки из платежной базы.

In [ ]:
import pandas as pd
import numpy as np

# Генерируем датафрейм, имитирующий сырую выгрузку платежей
data = {
    'payment_id': [101, 102, 103, 104, 105, 106, 107, 108],
    'user_id': [5541, 5542, 5543, 5541, 5544, 5545, 5546, 5547],
    'amount': [500, -100, 1500, 500, 0, 1000000, 750, 750],
    'status': ['success', 'refund', 'success', 'success', 'failed', 'success', 'success', 'failed'],
    'email': ['user1@gmail.com', 'user2@yandex.ru', 'VIP_user@mail.ru', 'user1@gmail.com', 'test_dev@cu.ru', 'test2_dev@cu.ru', 'user6@gmail.com', 'bad_email.com'],
    'date': ['2025-03-01', '2025-03-02', '2025-03-02', '2025-03-03', '2025-03-04', '2025-03-05', '2025-03-05', '2025-03-06']
}
df_payments = pd.DataFrame(data)
df_payments

### Задача 1. Первичный аудит (Разминка)
Маркетологи передали нам таблицу `df_payments`.

1. Выведите размерность датафрейма.
2. Посмотрите типы данных в таблице с помощью метода `.info()`.
3. Выведите описательные статистики для числовых колонок (`.describe()`).

In [ ]:
print("Размерность датафрейма:", df_payments.shape)
print("\n--- Информация о типах данных ---")
df_payments.info()
print("\n--- Описательные статистики ---")
df_payments.describe()

### Задача 2. Фильтрация и гигиена данных (Оператор И: &)
Из статистики выше мы увидели, что в данных есть отрицательные платежи (возвраты), нули (ошибки биллинга) и один огромный платеж на 1 млн (очевидно, тест разработчиков).

Напишите код, который отфильтрует `df_payments` и сохранит в новый датафрейм `df_clean` только те строки, где:
- Статус платежа строго `'success'`
- Сумма платежа (`amount`) строго больше 0
- Сумма платежа строго меньше 500 000

Выведите результат на экран.

In [ ]:
df_clean = df_payments[
    (df_payments['status'] == 'success') & 
    (df_payments['amount'] > 0) & 
    (df_payments['amount'] < 500000)
].copy()

df_clean

### Задача 3. Выборка через .loc и .iloc
Коллега из финотдела попросил выгрузить данные только о транзакциях с номерами строк от 2-й до 4-й (включительно) из очищенного датафрейма `df_clean`, при этом ему нужны только колонки `payment_id` и `amount`.

1. Сначала сделайте `df_clean.reset_index(drop=True, inplace=True)`, чтобы обновить индексы (иначе они останутся старыми).
2. Выполните срез с помощью метода `.loc`.
3. Выполните точно такой же срез с помощью метода `.iloc`.

In [ ]:
# 1. Сброс индексов
df_clean.reset_index(drop=True, inplace=True)

# 2. Срез через loc (индексы и имена колонок включительно)
res_loc = df_clean.loc[2:4, ['payment_id', 'amount']]

# 3. Срез через iloc (позиции строк от 2 до 4 -> это 2:5, и индексы колонок 0 и 2)
res_iloc = df_clean.iloc[2:5, [0, 2]]

print("Результат loc:\n", res_loc)
print("\nРезультат iloc:\n", res_iloc)

### Задача 4. Поиск проблемных транзакций (Оператор ИЛИ: |)
Отдел безопасности просит собрать лог "подозрительных" транзакций из изначального сырого датасета `df_payments`.

Сохраните в `df_suspicious` строки, в которых:
- Статус равен `'failed'` ИЛИ статус равен `'refund'`
- ИЛИ сумма транзакции `<= 0`.

In [ ]:
df_suspicious = df_payments[
    (df_payments['status'] == 'failed') |
    (df_payments['status'] == 'refund') |
    (df_payments['amount'] <= 0)
]

df_suspicious

### Задача 5. Арифметические операции со столбцами
Платежный шлюз забирает комиссию 5% за каждый успешный платеж. 

Создайте в `df_clean` новую колонку `net_amount` (чистая выручка), в которой будет лежать значение колонки `amount`, умноженное на 0.95.

In [ ]:
df_clean['net_amount'] = df_clean['amount'] * 0.95
df_clean.head()

### Задача 6. Работа со строками и отрицание (Оператор НЕ: ~)
Перед отправкой итогового списка успешных оплат маркетологам, нам нужно убедиться, что мы не отправляем рекламу нашим же разработчикам.

В `df_clean` отфильтруйте данные так, чтобы в таблице **НЕ** осталось почт с корпоративным доменом `@cu.ru`. 
Используйте оператор инверсии `~` и строковый метод `.str.contains()`.

In [ ]:
df_final = df_clean[~df_clean['email'].str.contains('@cu.ru')].copy()
df_final

### Задача 7. Разделение строк (split)
Нам нужно собрать аналитику по популярности почтовых сервисов среди платящих клиентов.

Создайте в `df_final` колонку `email_domain`, в которую с помощью метода `.str.split()` запишите только домен почты (то, что идет после знака `@`, например `gmail.com`).

In [ ]:
df_final['email_domain'] = df_final['email'].str.split('@').str[1]
df_final

### Задача 8. Работа с датами и индексами
В нашем итоговом датафрейме `df_final` дата хранится как текст.

1. Переведите колонку `date` в формат `datetime` с помощью `pd.to_datetime()`.
2. Создайте новую колонку `month`, в которой будет храниться номер месяца транзакции (используйте `.dt.month`).
3. Сделайте колонку `payment_id` новым индексом таблицы.

In [ ]:
df_final['date'] = pd.to_datetime(df_final['date'])
df_final['month'] = df_final['date'].dt.month

df_final.set_index('payment_id', inplace=True)
df_final

### Задача 9. Сортировка и поиск "Китов"
Отдел VIP-клиентов хочет подарить бесплатную подписку пользователю с самым большим чеком.

Отсортируйте `df_final` по размеру платежа (`amount`) по убыванию и выведите `user_id` человека на первой строчке.

In [ ]:
vip_user_id = df_final.sort_values('amount', ascending=False).head(1)['user_id']
print("VIP Пользователь:\n", vip_user_id)

### Задача 10. Визуальная проверка (семплирование)
Чтобы убедиться, что датафрейм собран корректно перед отправкой, выведите 2 случайные строки из `df_final` с помощью метода `.sample()`.

In [ ]:
df_final.sample(2)